In [1]:
import pandas as pd
import sklearn

In [2]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error


In [3]:
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID']
df=pd.read_parquet("yellow_tripdata_2023-01.parquet", columns=cols)

In [4]:
#Q1: How many columns are there in the dataset?
#len(df.columns)

In [5]:
#Q2: What's the standard deviation of the trips duration in January?
duration = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
duration = duration.dt.total_seconds() / 60
#duration.describe()

In [6]:
#Q3: There are some outliers. Let's remove them and keep only the records where the duration was between 1 and 60 minutes (inclusive). 
# What fraction of the records left after you dropped the outliers?
duration = duration[(duration >= 1) & (duration <= 60)]
#len(duration) / len(df)

In [7]:
#Q4: Apply one-hot encoding to the pickup and dropoff location IDs. We'll use only these two features for our model.
# What's the dimensionality of this matrix (number of columns)?
df['duration'] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
df = df[(df.duration >= 1) & (df.duration <= 60)]

categorical = ['PULocationID', 'DOLocationID']

df[categorical] = df[categorical].astype(str)
train_dicts = df[categorical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

#print(X_train.shape)

In [ ]:
#Q5: What's the RMSE on train?
#target = 'duration'
#y_train = df[target].values

lr = LinearRegression()
#lr.fit(X_train, y_train)

#y_pred = lr.predict(X_train)

#root_mean_squared_error(y_train, y_pred)

: 

In [ ]:
#Q6: What's the RMSE on validation?
df_val = pd.read_parquet("yellow_tripdata_2023-02.parquet")

df_val['duration'] = (df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime).dt.total_seconds() / 60
df_val = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)]

df_val[categorical] = df_val[categorical].astype(str).astype(object)
val_dicts = df_val[categorical].to_dict(orient='records')

X_val = dv.transform(val_dicts)
y_val = df_val['duration'].values

y_val_pred = lr.predict(X_val)
root_mean_squared_error(y_val, y_val_pred)

In [3]:
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID']
categorical = ['PULocationID', 'DOLocationID']

def prepare(path):
    df = pd.read_parquet(path, columns=cols)
    df['duration'] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    df[categorical] = df[categorical].astype(str).astype(object)
    y = df['duration'].values
    dicts = df[categorical].to_dict(orient='records')
    return dicts, y

In [6]:
# Training (January)
train_dicts, y_train = prepare('yellow_tripdata_2023-01.parquet')
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
del train_dicts
#print(X_train.shape)          # expect 515 columns

lr = LinearRegression().fit(X_train, y_train)
#print(root_mean_squared_error(y_train, lr.predict(X_train)))   # Q5

In [8]:
# Validation (February)
val_dicts, y_val = prepare('yellow_tripdata_2023-02.parquet')
X_val = dv.transform(val_dicts)
del val_dicts
print(root_mean_squared_error(y_val, lr.predict(X_val)))       # Q6

7.811819098973374
